# 02 · Detector de patrones atípicos: autoencoder (F3, H1)

**ObservaCompras IA** · MVP · Aplicaciones de Inteligencia Artificial (MTI-493), UTFSM  
Profesor: Sr. Werner Creixell · Alumnos: Alejandra Cornejo Jara y Leoncio Cornejo Miranda

Aprendizaje **no supervisado** (Sesión 2): el autoencoder aprende a reconstruir el perfil de compra típico; el error de reconstrucción mide la atipicidad y su desglose por variable es la explicación.

* Arquitectura: Dense 16 → 4 (latente) → 16 → 10, ReLU en capas ocultas y sigmoide en la salida.
* Pérdida `mse` (regresión a valores en [0, 1]); optimizador RMSprop.
* Validación K-fold con K = 5 (pocos cientos de organismos), early stopping (patience = 20).
* Búsqueda: learning rate ∈ {1e-2, 1e-3, 1e-4} × batch size ∈ {8, 16, 32}; hasta 500 epochs.
* Modelo final entrenado con todo 2023 por el número de epochs óptimo (como en Boston Housing).

> **Tiempo estimado:** la búsqueda completa son 45 entrenamientos (9 combinaciones × 5 folds) más 6 finales; en CPU toma entre 40 y 60 minutos. La red es pequeña, por lo que la GPU no acelera mucho.

In [ ]:
# Preparación del entorno (Google Colab o local)
import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # El repositorio se clona en Drive para que datos y modelos persistan entre cuadernos
    RAIZ = "/content/drive/MyDrive/observacompras-ia"
    if not os.path.exists(RAIZ):
        subprocess.run(["git", "clone", "https://github.com/lcornejom/observacompras-ia", RAIZ], check=True)
    os.chdir(RAIZ)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Modo de periodos: "anio" (2023 y 2024 completos, como en el documento) o
# "demo" (solo un semestre: 1er trimestre = "2023", 2º trimestre = "2024")
MODO = "anio"

from observacompras import config as C
print("Raíz del proyecto:", RAIZ)

In [ ]:
import inspect
from observacompras import autoencoder as AE
print(inspect.getsource(AE.construir_autoencoder))
AE.construir_autoencoder().summary()

## Entrenamiento, puntajes y evaluación de H1

La evaluación inyecta 20 organismos artificiales (5 tipos × 4) y mide qué fracción queda en el 5% superior del puntaje, frente a la línea base de puntaje z por variable.

In [ ]:
from observacompras import pipeline_ae
import json
resultado = pipeline_ae.ejecutar(verbose=0)
print(json.dumps(resultado, indent=2, ensure_ascii=False))

## Curvas de pérdida y búsqueda de hiperparámetros

In [ ]:
from IPython.display import Image, display
for f in ['ae_curva_perdida.png', 'ae_busqueda.png', 'ae_recall_por_tipo.png']:
    display(Image(str(C.DIR_FIGURAS / f)))

In [ ]:
import pandas as pd
tabla = pd.read_csv(C.DIR_REPORTES / 'ae_busqueda_hiperparametros.csv')
display(tabla.groupby(['learning_rate', 'batch_size'])[['mejor_val_loss', 'mejor_epoch']].mean().round(5))

## Ranking de organismos atípicos del año de evaluación

In [ ]:
punt = pd.read_parquet(C.DIR_PROCESADOS / 'puntajes_atipicidad.parquet')
top = punt[punt.anio == C.ANIO_PRUEBA].nsmallest(10, 'ranking')
pd.set_option('display.max_colwidth', 200)
display(top[['ranking', 'organismo_nombre', 'puntaje', 'percentil', 'explicacion']])

In [ ]:
display(pd.read_csv(C.DIR_REPORTES / 'ae_anomalias_sinteticas.csv'))